# Долгосрочная память аптечного AI-ассистента · этап 2: LLM-шаги на Qwen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/serge3emskov/pharma-ltm-assistant/blob/main/notebooks/02_llm_pipeline.ipynb)

На этапе 1 LLM-шаги были замоканы. Здесь их выполняет **Qwen3-4B-Instruct-2507** (4-bit, Colab T4):

| Шаг | Где | Что делает модель | Что делает код |
|---|---|---|---|
| Классификатор разделов | write-path | какие разделы затронуты в диалоге | отбрасывает неизвестные разделы; при сбое — все разделы |
| Extractor | write-path | события по **одному** разделу, с правилами только этого раздела | повтор при битом JSON, отсечение чужих разделов, валидация, нормализация, state machine |
| Классификатор запроса | read-path | тип запроса и субъект | отбрасывает неизвестные типы; при сбое — правила |
| Ответ | — | ответ покупателю по контексту | собирает контекст с пометкой источника и списком «уточнить» |

**Перед запуском:** Среда выполнения → Сменить среду выполнения → **T4 GPU**. Весь прогон занимает около 15 минут, повторные прогоны берут ответы из кэша.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/serge3emskov/pharma-ltm-assistant.git"

if IN_COLAB:
    if os.path.exists("pharma-ltm-assistant"):
        subprocess.run(["git", "-C", "pharma-ltm-assistant", "pull", "-q"], check=True)  # подтянуть свежий код
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = os.path.abspath("pharma-ltm-assistant")
    for req in ("requirements.txt", "requirements-llm.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/{req}"], check=True)
else:
    ROOT = os.path.abspath("..")

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# После git pull Python не перечитывает уже импортированный пакет — выгружаем старую версию
for name in [m for m in sys.modules if m == "pharmaltm" or m.startswith("pharmaltm.")]:
    del sys.modules[name]
import pharmaltm

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"pharmaltm {pharmaltm.__version__} · commit {commit or 'нет git'}")

## 1. Модель

`BACKEND` выбирает бэкенд:
* `"qwen"` — Qwen в этом ноутбуке через transformers (по умолчанию);
* `"server"` — OpenAI-совместимый сервер: vLLM, Ollama (`http://localhost:11434/v1`), LM Studio; адрес и модель — в переменных `LLM_BASE_URL` и `LLM_MODEL`;
* `"gold"` — эталонные ответы dev-набора, чтобы проверить ноутбук без GPU.

Все ответы модели кэшируются в `runs/llm_cache.jsonl`: повторный запуск ячеек не тратит GPU.

In [ ]:
import json, time
from datetime import datetime, timedelta, timezone

import pandas as pd
from IPython.display import Markdown, display

from pharmaltm import (CachedLLM, MemoryStore, OpenAICompatLLM, PharmacyAssistant, QueryClassifier,
                       QwenTransformersLLM, Section)
from pharmaltm.devcheck import (GoldLLM, load_dev, run_query_dev, run_write_dev, summarize_queries,
                                summarize_write)
from pharmaltm.prompts import extractor_messages

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

BACKEND = os.environ.get("PHARMALTM_BACKEND", "qwen")
RUN_TAG = "run3"  # метка прогона: попадает в имя файла отчёта
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
RUNS = os.path.join(ROOT, "runs")
os.makedirs(RUNS, exist_ok=True)

DIALOGS, QUERIES = load_dev()

t0 = time.time()
if BACKEND == "qwen":
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет — включите T4")
    # Повторный запуск ячейки не грузит вторую копию модели: на T4 это нехватка памяти
    prev = globals().get("base")
    base = prev if getattr(prev, "name", "").startswith(MODEL_ID) else QwenTransformersLLM(MODEL_ID, load_in_4bit=True)
elif BACKEND == "server":
    base = OpenAICompatLLM(os.environ["LLM_BASE_URL"], os.environ.get("LLM_MODEL", "hf.co/unsloth/Qwen3-4B-Instruct-2507-GGUF:Q4_K_M"))
else:
    base = GoldLLM(DIALOGS, QUERIES)
llm = CachedLLM(base, os.path.join(RUNS, "llm_cache.jsonl")) if BACKEND != "gold" else base
print(f"backend={BACKEND} model={llm.name} загрузка {time.time() - t0:.0f} с")

## 2. Один вызов extractor-а

Сырой ответ модели до какой-либо обработки кодом.

In [ ]:
turns = [{"role": "user", "content": "У меня гипертония, пью Энап 5 мг по утрам. Аллергий у меня нет."}]
t0 = time.time()
raw = llm.chat(extractor_messages(Section.MEDICATIONS, turns), max_new_tokens=512, json_mode=True,
               meta={"task": "extract", "section": "medications"})
print(f"{time.time() - t0:.1f} с\n{raw}")

## 3. Write-path на dev-наборе

17 диалогов с эталонами (d17 добавлен по сбою живого диалога в прогоне 2). Для каждого: свежая память (иногда с предысторией) → классификатор разделов → extractor по разделам → код. Сравнивается **итоговое состояние памяти**: ключ факта после нормализации и статус после state machine, а не дословный JSON. Метрика разделов — по ответу LLM-классификатора; `safety_net` — разделы, которые добавил код по справочнику препаратов.

In [ ]:
t0 = time.time()
write_results = run_write_dev(llm, DIALOGS)
write_summary = summarize_write(write_results, DIALOGS)
print(f"{time.time() - t0:.0f} с")
display(pd.json_normalize(write_summary).T.rename(columns={0: "значение"}))

In [ ]:
pd.DataFrame([r.row() for r in write_results])

Разбор ошибок: что ответила модель по каждому разделу в диалогах, где итоговая память не совпала с эталоном.

In [ ]:
for r in write_results:
    if r.ok:
        continue
    display(Markdown(f"**{r.id}** · {r.note}  \nпропущено: `{sorted(r.missing_facts | r.missing_states)}`  \n"
                     f"лишнее: `{sorted(r.extra_facts | r.extra_states)}`  \nошибки атрибутов: `{r.data_errors}`"))
    for c in r.write.calls:
        print(f"  [{c.task}:{c.section or '-'}] ok={c.ok} попыток={c.attempts} {c.error or ''}\n    {c.raw[:600]}")
    for rej in r.write.report.rejected:
        print("  ОТКЛОНЕНО валидацией:", rej.errors, rej.raw)

## 4. Классификатор запроса: Qwen против правил

In [ ]:
qc = QueryClassifier(llm)
t0 = time.time()
q_llm = run_query_dev(qc, QUERIES)
q_rules = run_query_dev(None, QUERIES)
print(f"{time.time() - t0:.0f} с")
display(pd.DataFrame({"Qwen": summarize_queries(q_llm), "правила": summarize_queries(q_rules)}))

In [ ]:
pd.DataFrame([{**a.row(), "rules_pred": b.row()["pred"]} for a, b in zip(q_llm, q_rules)
              if not (a.types_ok and a.subject_ok) or not b.types_ok])

## 5. Живой сценарий: две сессии с паузой

Read-path работает на каждое сообщение, write-path — после окончания фрагмента диалога (`close()`), как фоновая запись в статье. Через неделю ассистент отвечает с учётом того, что покупатель рассказал в первый раз.

In [ ]:
class Clock:
    def __init__(self):
        self.now = datetime(2026, 10, 1, 10, 0, tzinfo=timezone.utc)
    def __call__(self):
        return self.now
    def advance(self, days):
        self.now += timedelta(days=days)


clock = Clock()
bot = PharmacyAssistant(llm, MemoryStore(), clock=clock)
USER = "buyer-001"


def chat(session, text):
    display(Markdown(f"**Покупатель:** {text}"))
    answer = session.say(text)
    read = session.replies[-1].read
    display(Markdown(f"*тип запроса: {[q.value for q in read.classification.query_types]}, "
                     f"субъект: {read.classification.subject}, правила: {read.fallback_used}*  \n"
                     f"**Ассистент:** {answer}"))


s1 = bot.session(USER, "session-1")
chat(s1, "Здравствуйте! У меня гипертония, пью Лозап 50 по утрам. На пенициллин аллергия — была крапивница.")
chat(s1, "И ещё: я кормлю грудью. Что можно от насморка?")
w1 = s1.close()
display(Markdown("**Write-path после сессии.** Модель видит реплики покупателя и только вопросы ассистента:"))
for t in w1.turns:
    print(f"  {t['role']}: {t['content']}")
display(pd.DataFrame([w1.summary()]))
pd.DataFrame([{"section": f.section.value, "key": f.key, "status": f.status, "data": f.data, "said": f.raw_names}
              for f in bot.store.list_facts(USER)])

In [ ]:
clock.advance(7)
s2 = bot.session(USER, "session-2")
chat(s2, "Добрый день. Голова сильно болит с утра, что можно выпить?")
print("\nКонтекст, который получила модель:\n")
print(s2.replies[-1].read.bundle.render())
chat(s2, "А сколько таблеток парацетамола можно выпить за день?")  # дозы — только из инструкции

## 6. Отчёт о прогоне

Сводка, строки по диалогам и запросам и сырые ответы модели по ошибкам сохраняются в JSON. Этот файл можно прислать для разбора промптов.

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M")
report = {
    "model": llm.name,
    "backend": BACKEND,
    "run_tag": RUN_TAG,
    "write_summary": write_summary,
    "write_rows": [r.row() for r in write_results],
    "write_failures": [r.write.as_dict() for r in write_results if not r.ok],
    "query_summary": {"llm": summarize_queries(q_llm), "rules": summarize_queries(q_rules)},
    "query_rows": [{**r.row(), "raw": r.raw} for r in q_llm],
    "chat": [rep.as_dict() for s in (s1, s2) for rep in s.replies],
    "chat_write": [w1.as_dict()],
}
path = os.path.join(RUNS, f"stage2_{RUN_TAG}_{BACKEND}_{stamp}.json")
with open(path, "w", encoding="utf-8") as fh:
    json.dump(report, fh, ensure_ascii=False, indent=2, default=list)
print("сохранено:", path)
if IN_COLAB:
    from google.colab import files
    files.download(path)

## Что дальше · этап 3

1. Golden set на 80–100 диалогов: перефразы, опечатки, падежи, смешанные субъекты, «ловушки» (препарат только в вопросе).
2. Метрики по слоям, как в статье: классификатор разделов, extractor по полям, переходы состояния, retrieval, E2E.
3. Сравнение Qwen3-4B и более крупной модели (7–8B) на тех же диалогах.

> Демонстрационный проект. Данные синтетические, ответы ассистента не являются медицинской рекомендацией.